In [150]:
import os
import pickle
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [117]:
# from google.colab import drive
# drive.mount('/content/drive')

In [118]:
# Charger les données

data_path = "../data/processed"
file_name = 'dataset_final.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [119]:
# Charger un seul embedding

embeddings_path = "../data/embeddings"
# file_name = 'emb_sbert_multi.pkl'

# with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#     embedding = pickle.load(f)

In [120]:
# Charger tous les embeddings

# embeddings_path = "../data/embeddings"
# embeddings = {}

# for file_name in os.listdir(embeddings_path):
#     with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#         emb_name = file_name.split('.')[0]
#         embeddings[emb_name] = pickle.load(f)

### BERTopic

In [233]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
from hdbscan import HDBSCAN

In [122]:
sentences = df["clean_comment"].tolist()

file_name_mutli = 'emb_sbert_multi.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [123]:
print(len(sentences))
print(embedding_multi.shape)
print(embedding_fr.shape)
print(embedding_multi2.shape)

15077
(15077, 384)
(15077, 768)
(15077, 768)


In [238]:
product_words = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french") + product_words,
        ngram_range=(1, 3),
        #min_df=2,
        max_df=0.95
    )
    # ctfidf_model = ClassTfidfTransformer()

    hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=3
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        nr_topics="auto"
        # ctfidf_model=ctfidf_model
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding)

    topic_model.reduce_topics(sentences, nr_topics=30)

    models[name] = topic_model

### Evaluation

#### Diversité

In [239]:
from itertools import chain

def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.6448275862068965,
 'multilingue': 0.7344827586206897,
 'multilingue_2': 0.7862068965517242}

#### Score de cohérence

In [240]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def coherence_score(topic_model, documents, top_n=10):
    """
    Calcule le score de cohérence classique c_v pour un modèle BERTopic.
    """
    # print(type(model.get_topics()))
    # print(model.get_topics()[0])
    # Récupère les topics (liste de mots par topic)
    topics = topic_model.get_topics()
    topic_words = []
    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        words = []
        for w, s in word_scores[:top_n]:
            w = str(w)
            words.append(w)
        topic_words.append(words)
    # print(topic_words[:3])

    # Préparer le corpus pour Gensim
    tokenized_docs = [doc.lower().split() for doc in documents]  # tokenisation simple
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

    # Calcul du score
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return cm.get_coherence()

# Exemple
for name, model in models.items():
    score = coherence_score(model, sentences, top_n=10)
    print(f"{name} - Coherence c_v: {score:.4f}")

# score = coherence_score(models["multilingue_2"], sentences, top_n=10)
# print(f"{name} - Coherence c_v: {score:.4f}")

français - Coherence c_v: 0.5387
multilingue - Coherence c_v: 0.4318
multilingue_2 - Coherence c_v: 0.6073


#### Embedding-based coherence score

In [242]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.3950
multilingue - Embedding-based coherence: 0.3404
multilingue_2 - Embedding-based coherence: 0.4360


In [246]:
for name, model in models.items():
    with open(f"../models/topic_modeling/bertopic_{name}.pkl", "wb") as f:
        pickle.dump(model, f)

### Visualisation

In [8]:
# CHARGER LES MODELES ENREGISTRES
# models = {}
# for name in ["français", "multilingue", "multilingue_2"]:
#     with open(os.path.join("../models/topic_modeling/", f"bertopic_{name}.pkl"), 'rb') as f:
#         models[name] = pickle.load(f)

In [247]:
all_topics_words = {}
topic_model = models["multilingue_2"]

for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue  # ignorer les outliers
    all_topics_words[topic_id] = [word for word, _ in topic_model.get_topic(topic_id)]

all_topics_words

{0: ['commande',
  'plus',
  'livraison',
  'colis',
  'site',
  'service',
  'vente',
  'client',
  'bien',
  'remboursement'],
 1: ['conforme',
  'livraison',
  'description',
  'rapide',
  'attentes',
  'conforme description',
  'produit conforme',
  'conformes',
  'produit',
  'prévue'],
 2: ['conforme',
  'attentes',
  'conforme attentes',
  'description',
  'produit conforme',
  'correspond',
  'attentes produit',
  'produit',
  'conforme description',
  'conforme commande'],
 3: ['rien dire',
  'rien',
  'dire',
  'rien redire',
  'redire',
  'parfait',
  'parfait rien',
  'rien dire rien',
  'dire rien',
  'parfait rien dire'],
 4: ['satisfaite',
  'très satisfaite',
  'contente',
  'achat très',
  'achat',
  'satisfaite achat',
  'achat très satisfaite',
  'très contente',
  'satisfaite achat très',
  'très satisfaite achat'],
 5: ['qualité',
  'déçue',
  'déçue qualité',
  'déçu',
  'photo',
  'déçu qualité',
  'correspondent',
  'articles',
  'photos',
  'correspond'],
 6: [

In [248]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4802,-1_livraison_commande_qualité_bien,"[livraison, commande, qualité, bien, produit, ...",[livraison trop longue et produit de mauvaise ...
1,0,7300,0_commande_plus_colis_site,"[commande, plus, colis, site, service, vente, ...",[j'avais déjà écrit un avis sur vente-privée i...
2,1,427,1_article_carton_cassé_peu,"[article, carton, cassé, peu, livraison, manqu...","[il manque un article, il manque un article, i..."
3,2,374,2_livraison_peu_livraison peu_long,"[livraison, peu, livraison peu, long, peu long...","[délai de livraison un peu long, le délai de l..."
4,3,297,3_satisfaite_rapide_parfait_livraison,"[satisfaite, rapide, parfait, livraison, comma...","[satisfaite de ma commande, satisfaite de la c..."
5,4,202,4_rien dire_rien_dire_parfait,"[rien dire, rien, dire, parfait, merci, rien r...","[parfait , rien à dire, parfait rien a dire ....."
6,5,187,5_rapide_livraison_livraison rapide_produit,"[rapide, livraison, livraison rapide, produit,...","[bon produit et livraison rapide, bon produit ..."
7,6,166,6_satisfaite_très satisfaite_contente_commande...,"[satisfaite, très satisfaite, contente, comman...","[très satisfaite de la commande ., très satisf..."
8,7,161,7_satisfaite_très satisfaite_contente_merci,"[satisfaite, très satisfaite, contente, merci,...","[merci je suis très satisfaite, j ai été très ..."
9,8,138,8_qualité_bonne qualité_bonne_belle,"[qualité, bonne qualité, bonne, belle, qualité...","[déçue par la qualité du sac, déçue par la qua..."


In [249]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,5429,-1_commande_plus_colis_service,"[commande, plus, colis, service, livraison, si...",[voici mon expérience avec vente-privée . depu...
1,0,7498,0_livraison_commande_très_plus,"[livraison, commande, très, plus, site, colis,...",[une petite valise cabine delsey commandée et ...
2,1,449,1_très_plus_commandé_commande,"[très, plus, commandé, commande, pointure, tai...",[en date du 15/05/2020 je commande une paire d...
3,2,276,2_photo_photos_couleur_conforme,"[photo, photos, couleur, conforme, conforme ph...","[produit conforme à la photo, produit pas conf..."
4,3,225,3_très_showroom_service_satisfaite,"[très, showroom, service, satisfaite, showroom...","[très satisfaite du service showroom showroom,..."
5,4,162,4_qualité_produits_déçu_produit,"[qualité, produits, déçu, produit, déçue, très...","[je suis déçue de la qualité, très déçue par l..."
6,5,127,5_or_bagues_avoir_pierre,"[or, bagues, avoir, pierre, très, reçu, plus, ...",[bonjour je vous fais par de mon expérience d ...
7,6,117,6_jours_plus_mois_vente,"[jours, plus, mois, vente, prs, après, batteri...",[j'ai acheté un forfait free mobile avec un té...
8,7,81,7_attentes_conforme attentes_correspond attent...,"[attentes, conforme attentes, correspond atten...","[conforme a mes attentes, conforme à mes atten..."
9,8,77,8_chaises_chaise_fauteuil_plus,"[chaises, chaise, fauteuil, plus, commandé, ve...",[ayant commandé quatres chaises de salle à man...


In [250]:
# topics trouvés
models["multilingue_2"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4875,-1_livraison_commande_qualité_plus,"[livraison, commande, qualité, plus, produit, ...",[déçu par la qualité des lunettes et livraison...
1,0,8843,0_commande_plus_livraison_colis,"[commande, plus, livraison, colis, site, servi...","[premier et dernier achat de chez eux , plus j..."
2,1,179,1_conforme_livraison_description_rapide,"[conforme, livraison, description, rapide, att...","[livraison rapide , produit conforme à mes att..."
3,2,172,2_conforme_attentes_conforme attentes_description,"[conforme, attentes, conforme attentes, descri...","[conforme a mes attentes, conforme à mes atten..."
4,3,139,3_rien dire_rien_dire_rien redire,"[rien dire, rien, dire, rien redire, redire, p...","[parfait rien a dire ..., parfait , rien a dir..."
5,4,99,4_satisfaite_très satisfaite_contente_achat très,"[satisfaite, très satisfaite, contente, achat ...","[très satisfaite de mon achat ., je suis très ..."
6,5,92,5_qualité_déçue_déçue qualité_déçu,"[qualité, déçue, déçue qualité, déçu, photo, d...","[déçue par la qualité des vêtements, je suis d..."
7,6,61,6_trop_joli_petit_très joli,"[trop, joli, petit, très joli, trop petit, jol...","[les articles étaient beaucoup trop petit ., t..."
8,7,56,7_rapide_livraison rapide_produit_livraison,"[rapide, livraison rapide, produit, livraison,...",[article de très bonne qualité et livraison ra...
9,8,54,8_long_trop long_trop_livraison,"[long, trop long, trop, livraison, livraison t...","[délai de livraison trop long, le délai de liv..."


In [251]:
# Mots-clés associés à un topic
models["français"].get_topic(0)

[('commande', np.float64(0.010340701984429202)),
 ('plus', np.float64(0.00986959838147149)),
 ('colis', np.float64(0.007717754654670451)),
 ('site', np.float64(0.007657288166145758)),
 ('service', np.float64(0.007449170381609198)),
 ('vente', np.float64(0.0067841269904930415)),
 ('livraison', np.float64(0.00670084095999104)),
 ('client', np.float64(0.00665961608731454)),
 ('remboursement', np.float64(0.0059745462739572855)),
 ('après', np.float64(0.005613513495396216))]

In [252]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(0)

[('livraison', np.float64(0.01209649178161709)),
 ('commande', np.float64(0.01157781764070824)),
 ('très', np.float64(0.011430530478093958)),
 ('plus', np.float64(0.00938488756488948)),
 ('site', np.float64(0.007777954416698765)),
 ('colis', np.float64(0.007690737782841499)),
 ('bien', np.float64(0.007397506768599091)),
 ('service', np.float64(0.006689339871293234)),
 ('tout', np.float64(0.006139802726551881)),
 ('qualité', np.float64(0.006094333420782947))]

In [253]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
models["français"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_hierarchy()

In [ ]:
models["multilingue"].visualize_hierarchy()